## Experiment Set 3: Dynamic Batching, max batch size 4

Dynamic batching, poisson arrivals, asynchronous calls, varying arrival rates

To edit the model configuration:

``` bash
# runs on node-serve-system
nano ~/serve-system-chi/models/food_classifier_onnx/config.pbtxt
```

Current config:

``` bash
name: "food_classifier_onnx"
backend: "onnxruntime"
max_batch_size: 4
input [
  {
    name: "input"  # has to match ONNX model's input name
    data_type: TYPE_FP32
    dims: [3, 224, 224]  # has to match ONNX input shape
  }
]
output [
  {
    name: "output"  # has to match ONNX model output name
    data_type: TYPE_FP32  # output is a list of probabilities
    dims: [11]  #
  }
]
  instance_group [
    {
      count: 1
      kind: KIND_GPU
      gpus: [ 0 ]
    }
]

dynamic_batching {
    preferred_batch_size: [4]
    max_queue_delay_microseconds: 100
}

```
Save the file (use Ctrl+O then Enter, then Ctrl+X).

Re-build the container image with this change:

``` bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml build triton_server
```

and then bring the server back up:

``` bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml up triton_server --force-recreate -d
```

and use

``` bash
# runs on node-serve-system
docker logs triton_server
```

to make sure the server comes up and is ready.

Before we benchmark this service again, let’s get some pre-benchmark stats about how many requests have been served, broken down by batch size. (If you’ve just restarted the server, it would be zero!)

In [2]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 20:100:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_4_results/batch_4_20-100.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 100 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 20 inference requests per second
  Client: 
    Request count: 335
    Throughput: 18.4884 infer/sec
    Avg latency: 9130 usec (standard deviation 2024 usec)
    p50 latency: 8600 usec
    p90 latency: 10296 usec
    p95 latency: 12301 usec
    p99 latency: 16701 usec
    Avg HTTP time: 8988 usec (send/recv 864 usec + response wait 8124 usec)
  Server: 
    Inference count: 335
    Execution count: 331
    Successful request count: 335
    Avg request latency: 6348 usec (overhead 40 usec + queue 559 usec + compute input 158 usec + compute infer 5568 usec + compute output 22 usec)
Request Rate: 40 inference requests per second
  

In [3]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_4_results/stats_20-100.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778451229444,
            "inference_count": 10685,
            "execution_count": 10194,
            "inference_stats": {
                "success": {
                    "count": 10685,
                    "ns": 1802080666467
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 10685,
                    "ns": 1685841256707
                },
                "compute_input": {
                    "count": 10685,
                    "ns": 2038361081
                },
                "compute_infer": {
                    "count": 10685,
                    "ns": 113601824933
                },
                "compute_output": {
                    "count": 10685,
                    "ns": 217715669
                },
       

In [4]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 120:200:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_4_results/batch_4_120-200.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 200 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 120 inference requests per second
  Client: 
    Request count: 2149
    Throughput: 116.455 infer/sec
    Avg latency: 9876 usec (standard deviation 3396 usec)
    p50 latency: 8790 usec
    p90 latency: 14098 usec
    p95 latency: 15643 usec
    p99 latency: 18559 usec
    Avg HTTP time: 9820 usec (send/recv 653 usec + response wait 9167 usec)
  Server: 
    Inference count: 2149
    Execution count: 1940
    Successful request count: 2149
    Avg request latency: 7222 usec (overhead 36 usec + queue 2106 usec + compute input 178 usec + compute infer 4882 usec + compute output 19 usec)
Request Rate: 140 inference requests per se

In [5]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_4_results/stats_120-200.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778451375372,
            "inference_count": 31526,
            "execution_count": 28562,
            "inference_stats": {
                "success": {
                    "count": 31526,
                    "ns": 1937821172473
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 31526,
                    "ns": 1729322272438
                },
                "compute_input": {
                    "count": 31526,
                    "ns": 5537198202
                },
                "compute_infer": {
                    "count": 31526,
                    "ns": 201403190034
                },
                "compute_output": {
                    "count": 31526,
                    "ns": 566178674
                },
       

In [6]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 220:300:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_4_results/batch_4_220-300.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 300 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 220 inference requests per second
  Client: 
    Request count: 4184
    Avg send request rate: 230.00 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 1.63% of the requests were delayed. 
    Throughput: 216.77 infer/sec
    Avg latency: 8972 usec (standard deviation 5745 usec)
    p50 latency: 7944 usec
    p90 latency: 12836 usec
    p95 latency: 14712 usec
    p99 latency: 24522 usec
    Avg HTTP time: 8824 usec (send/recv 737 usec + response wait 8087 usec)
  Server: 
    Inference count: 4184
    Execution count: 3540
    Successful request count: 4184
    Avg request latency: 537

In [7]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_4_results/stats_220-300.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778451621706,
            "inference_count": 68063,
            "execution_count": 58286,
            "inference_stats": {
                "success": {
                    "count": 68063,
                    "ns": 2128167230008
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 68063,
                    "ns": 1799815994377
                },
                "compute_input": {
                    "count": 68063,
                    "ns": 11814688731
                },
                "compute_infer": {
                    "count": 68063,
                    "ns": 313648315903
                },
                "compute_output": {
                    "count": 68063,
                    "ns": 1029404468
                },
     

In [8]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 320:400:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_4_results/batch_4_320-400.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 400 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 320 inference requests per second
  Client: 
    Request count: 7055
    Avg send request rate: 322.83 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 4.11% of the requests were delayed. 
    Throughput: 330.14 infer/sec
    Avg latency: 10285 usec (standard deviation 13461 usec)
    p50 latency: 8231 usec
    p90 latency: 13594 usec
    p95 latency: 19556 usec
    p99 latency: 49563 usec
    Avg HTTP time: 9840 usec (send/recv 793 usec + response wait 9047 usec)
  Server: 
    Inference count: 7055
    Execution count: 5259
    Successful request count: 7055
    Avg request latency: 5

In [9]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_4_results/stats_320-400.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778451873189,
            "inference_count": 155727,
            "execution_count": 119507,
            "inference_stats": {
                "success": {
                    "count": 155727,
                    "ns": 2651627735967
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 155727,
                    "ns": 2033262702687
                },
                "compute_input": {
                    "count": 155727,
                    "ns": 31943002889
                },
                "compute_infer": {
                    "count": 155727,
                    "ns": 580169319477
                },
                "compute_output": {
                    "count": 155727,
                    "ns": 2148953717
                }

In [10]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 420:500:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_4_results/batch_4_420-500.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 500 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 420 inference requests per second
  Client: 
    Request count: 9447
    Avg send request rate: 417.39 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 4.56% of the requests were delayed. 
    Throughput: 429.92 infer/sec
    Avg latency: 11586 usec (standard deviation 15034 usec)
    p50 latency: 9058 usec
    p90 latency: 14450 usec
    p95 latency: 26231 usec
    p99 latency: 60487 usec
    Avg HTTP time: 11221 usec (send/recv 821 usec + response wait 10400 usec)
  Server: 
    Inference count: 9446
    Execution count: 5993
    Successful request count: 9446
    Avg request latency:

In [11]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_4_results/stats_420-500.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778452088291,
            "inference_count": 244073,
            "execution_count": 170328,
            "inference_stats": {
                "success": {
                    "count": 244073,
                    "ns": 3278814393272
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 244073,
                    "ns": 2345405588619
                },
                "compute_input": {
                    "count": 244073,
                    "ns": 58285311084
                },
                "compute_infer": {
                    "count": 244073,
                    "ns": 865012031382
                },
                "compute_output": {
                    "count": 244073,
                    "ns": 3364775576
                }

In [12]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 520:600:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_4_results/batch_4_520-600.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 600 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 520 inference requests per second
  Client: 
    Request count: 10876
    Avg send request rate: 533.53 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 3.76% of the requests were delayed. 
    Throughput: 520.86 infer/sec
    Avg latency: 12290 usec (standard deviation 11982 usec)
    p50 latency: 10275 usec
    p90 latency: 16753 usec
    p95 latency: 26383 usec
    p99 latency: 48928 usec
    Avg HTTP time: 12068 usec (send/recv 499 usec + response wait 11569 usec)
  Server: 
    Inference count: 10875
    Execution count: 5435
    Successful request count: 10875
    Avg request late

In [13]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_4_results/stats_520-600.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778452377301,
            "inference_count": 403660,
            "execution_count": 245868,
            "inference_stats": {
                "success": {
                    "count": 403660,
                    "ns": 4648251398210
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 403660,
                    "ns": 3118376812240
                },
                "compute_input": {
                    "count": 403660,
                    "ns": 116865140741
                },
                "compute_infer": {
                    "count": 403660,
                    "ns": 1395169753142
                },
                "compute_output": {
                    "count": 403660,
                    "ns": 5693227403
               

In [14]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 620:700:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_4_results/batch_4_620-700.txt 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 700 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 620 inference requests per second
  Client: 
    Request count: 13383
    Avg send request rate: 633.71 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 3.99% of the requests were delayed. 
    Throughput: 635.36 infer/sec
    Avg latency: 13529 usec (standard deviation 14530 usec)
    p50 latency: 10303 usec
    p90 latency: 19569 usec
    p95 latency: 35017 usec
    p99 latency: 62741 usec
    Avg HTTP time: 13292 usec (send/recv 612 usec + response wait 12680 usec)
  Server: 
    Inference count: 13383
    Execution count: 5795
    Successful request count: 13383
    Avg request late

In [15]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_4_results/stats_620-700.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778452721483,
            "inference_count": 626995,
            "execution_count": 337283,
            "inference_stats": {
                "success": {
                    "count": 626995,
                    "ns": 7131492417690
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 626995,
                    "ns": 4760210739993
                },
                "compute_input": {
                    "count": 626995,
                    "ns": 207530954610
                },
                "compute_infer": {
                    "count": 626995,
                    "ns": 2134507357567
                },
                "compute_output": {
                    "count": 626995,
                    "ns": 8986540127
               

In [16]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 720:800:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_4_results/batch_4_720-800.txt 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 800 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 720 inference requests per second
  Client: 
    Request count: 18333
    Avg send request rate: 720.74 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 7.79% of the requests were delayed. 
    Throughput: 724.66 infer/sec
    Avg latency: 28553 usec (standard deviation 27352 usec)
    p50 latency: 11729 usec
    p90 latency: 88726 usec
    p95 latency: 117573 usec
    p99 latency: 169823 usec
    Avg HTTP time: 28130 usec (send/recv 2318 usec + response wait 25812 usec)
  Server: 
    Inference count: 18333
    Execution count: 6774
    Successful request count: 18333
    Avg request l

In [17]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_4_results/stats_720-800.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778453073665,
            "inference_count": 888655,
            "execution_count": 426298,
            "inference_stats": {
                "success": {
                    "count": 888655,
                    "ns": 12320171730046
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 888655,
                    "ns": 8922397551816
                },
                "compute_input": {
                    "count": 888655,
                    "ns": 332400805838
                },
                "compute_infer": {
                    "count": 888655,
                    "ns": 3021588834588
                },
                "compute_output": {
                    "count": 888655,
                    "ns": 13021470398
             

In [18]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 820:900:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_4_results/batch_4_820-900.txt 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 900 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 820 inference requests per second
  Client: 
    Request count: 20107
    Avg send request rate: 805.34 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 7.20% of the requests were delayed. 
    Throughput: 834.65 infer/sec
    Avg latency: 37907 usec (standard deviation 19333 usec)
    p50 latency: 17629 usec
    p90 latency: 107015 usec
    p95 latency: 139803 usec
    p99 latency: 192734 usec
    Avg HTTP time: 37598 usec (send/recv 2030 usec + response wait 35568 usec)
  Server: 
    Inference count: 20110
    Execution count: 6152
    Successful request count: 20110
    Avg request 

In [19]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_4_results/stats_820-900.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778453542248,
            "inference_count": 1263022,
            "execution_count": 534982,
            "inference_stats": {
                "success": {
                    "count": 1263022,
                    "ns": 30803791136754
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 1263022,
                    "ns": 25892841128853
                },
                "compute_input": {
                    "count": 1263022,
                    "ns": 528954420707
                },
                "compute_infer": {
                    "count": 1263022,
                    "ns": 4316164452528
                },
                "compute_output": {
                    "count": 1263022,
                    "ns": 19032772014
      

In [20]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 920:1000:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_4_results/batch_4_920-1000.txt 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 1000 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 920 inference requests per second
  Client: 
    Request count: 21795
    Avg send request rate: 985.92 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 6.57% of the requests were delayed. 
    Throughput: 933.23 infer/sec
    Avg latency: 73146 usec (standard deviation 27412 usec)
    p50 latency: 61792 usec
    p90 latency: 136434 usec
    p95 latency: 147254 usec
    p99 latency: 226221 usec
    Avg HTTP time: 72849 usec (send/recv 2356 usec + response wait 70493 usec)
  Server: 
    Inference count: 21791
    Execution count: 5687
    Successful request count: 21791
    Avg request

In [21]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_4_results/stats_920-1000.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778454419887,
            "inference_count": 1639713,
            "execution_count": 645545,
            "inference_stats": {
                "success": {
                    "count": 1639713,
                    "ns": 192665707573244
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 1639713,
                    "ns": 186205895321054
                },
                "compute_input": {
                    "count": 1639713,
                    "ns": 736756812088
                },
                "compute_infer": {
                    "count": 1639713,
                    "ns": 5634267511149
                },
                "compute_output": {
                    "count": 1639713,
                    "ns": 25370586147
    